In [ ]:
import xarray
import numpy as np

In [ ]:
sst_time = xarray.open_zarr("../../data/processed/sst/sst_2015_2024_025deg.zarr")["time"].values
sss_time = xarray.open_zarr("../../data/processed/sss/sss_2015_2024_025deg.zarr")["time"].values
print(np.array_equal(sst_time, sss_time))

In [ ]:
sst_ds = xarray.open_zarr("../../data/raw/sst/c3s_sst_2015_2024_005deg.zarr")
wind_ds = xarray.open_zarr("../../data/raw/wind/cmems_wind_2015_2024_0125deg_hourly.zarr")
sss_ds = xarray.open_zarr("../../data/raw/sss/multiobs_sss_my_2015_2024-12-15_0125deg.zarr")
sealevel_ds = xarray.open_zarr("../../data/raw/sealevel/cmems_sealevel_adt_ugos_vgos_2015_2024_0125deg.zarr")

In [ ]:
sst_ds.info()
sst_ds["analysed_sst"].chunks

In [ ]:
sss_ds.info()
sss_ds["sos"].chunks
sss_ds["depth"].values

In [ ]:
sealevel_ds.info()
sealevel_ds["adt"].chunks

In [ ]:
wind_ds.info()
wind_ds["eastward_wind"].chunks

In [ ]:
sst = sst_ds.analysed_sst

value = sst.sel(
    time="2015-01-01",
    latitude=15,
    longitude=80,
    method="nearest"
).compute().item()

value

In [ ]:
small_area = sst.sel(
    time="2015-01-01",
    latitude=slice(14, 16),
    longitude=slice(79, 81)
)

print(small_area.compute())

In [ ]:
# Plotting using matplotlib

import matplotlib.pyplot as plt

sst_day = sst_ds.analysed_sst.sel(time="2015-01-01")

sst_day.plot(
    x="longitude",
    y="latitude",
    figsize=(12, 6),
    cmap="turbo"
)

plt.show()

In [ ]:
sss_my = xarray.open_zarr("../../data/raw/sss/multiobs_sss_my_2015_2024-12-15_0125deg.zarr")["sos"]
sss_nrt = xarray.open_zarr("../../data/raw/sss/multiobs_sss_nrt_2024-12-16_2024-12-31_0125deg.zarr")["sos"]

print(np.array_equal(sss_my["latitude"].values, sss_nrt["latitude"].values))
print(np.array_equal(sss_my["longitude"].values, sss_nrt["longitude"].values))

In [ ]:
wind_ds.info()
wind_ds.chunks

In [ ]:
print(wind_ds.time.values[:30])
print(wind_ds.time.values[-30:])

print(
    wind_ds.time.diff("time").values[:100]
)

print(
    wind_ds["eastward_wind"].dtype,
    wind_ds["northward_wind"].dtype
)

In [ ]:
print(
    wind_ds.time.resample(time="1D")
    .count()
)

In [ ]:
sealevel_ds = xarray.open_zarr("../../data/raw/sealevel/cmems_sealevel_adt_ugos_vgos_2015_2024_0125deg.zarr")

In [ ]:
adt = sealevel_ds["adt"].astype("float32")
ugos = sealevel_ds["ugos"].astype("float32")
vgos = sealevel_ds["vgos"].astype("float32")

In [ ]:
adt_mask = adt.notnull().any(dim="time").compute()
ugos_mask = ugos.notnull().any(dim="time").compute()
vgos_mask = vgos.notnull().any(dim="time").compute()

print("ADT valid cells :", int(adt_mask.sum().item()))
print("UGOS valid cells:", int(ugos_mask.sum().item()))
print("VGOS valid cells:", int(vgos_mask.sum().item()))
print("UGOS ⊄ ADT mask:", int((ugos_mask & ~adt_mask).sum().item()))
print("VGOS ⊄ ADT mask:", int((vgos_mask & ~adt_mask).sum().item()))

In [ ]:
sealevel_ds.info()
sealevel_ds.chunks

In [ ]:
glorys_ds = xarray.open_zarr("../../data/raw/glorys/cmems_mod_glo_phy_my_0.083deg_P1D-m_thetao_45.00E-105.00E_5.00N-30.00N_0.49-1062.44m_2015-01-01T00-00-00-2015-12-31T00-00-00.zarr")

In [ ]:
glorys_ds.info()
glorys_ds.depth.values

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr

ROOT = Path("data/raw/glorys")
TARGET_DEPTHS = np.array([0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000], dtype=float)

for path in sorted(ROOT.glob("cmems_*.zarr")):
    print("\n" + "=" * 90)
    print(path.name)
    print("=" * 90)

    ds = xr.open_zarr(path, consolidated=True)
    thetao = ds["thetao"]

    print("\nDIMENSIONS")
    for dim, size in ds.sizes.items():
        print(f"  {dim:12s}: {size}")

    print("\nDATA TYPE / STORAGE")
    print(f"  thetao dtype       : {thetao.dtype}")
    print(f"  thetao chunks      : {thetao.chunks}")
    print(f"  thetao encoding    : {thetao.encoding}")

    print("\nTIME")
    print(f"  start              : {ds.time.values[0]}")
    print(f"  end                : {ds.time.values[-1]}")
    print(f"  timesteps          : {ds.sizes['time']}")
    print(f"  time resolution    : {np.diff(ds.time.values[:2])[0] if ds.sizes['time'] > 1 else 'N/A'}")

    print("\nSPATIAL GRID")
    lat = ds.latitude.values
    lon = ds.longitude.values

    print(f"  latitude range     : {lat.min()} to {lat.max()}")
    print(f"  longitude range    : {lon.min()} to {lon.max()}")
    print(f"  latitude points    : {len(lat)}")
    print(f"  longitude points   : {len(lon)}")
    print(f"  latitude spacing   : {np.median(np.diff(lat))}")
    print(f"  longitude spacing  : {np.median(np.diff(lon))}")
    print(f"  latitude regular   : {np.allclose(np.diff(lat), np.diff(lat)[0])}")
    print(f"  longitude regular  : {np.allclose(np.diff(lon), np.diff(lon)[0])}")
    print(f"  latitude monotonic : {np.all(np.diff(lat) > 0)}")
    print(f"  longitude monotonic: {np.all(np.diff(lon) > 0)}")

    print("\nDEPTH LEVELS")
    depth = ds.depth.values
    print(f"  number of levels   : {len(depth)}")
    print(f"  minimum depth      : {depth.min()} m")
    print(f"  maximum depth      : {depth.max()} m")
    print(f"  levels:")
    print("   ", np.array2string(depth, precision=3, separator=", "))

    print("\nTARGET DEPTH MATCHING")
    for target in TARGET_DEPTHS:
        exact = np.where(np.isclose(depth, target, atol=1e-5))[0]

        if len(exact):
            print(f"  {target:6.1f} m -> exact level {depth[exact[0]]:.3f} m")
        elif target < depth.min():
            print(f"  {target:6.1f} m -> below minimum GLORYS depth")
        elif target > depth.max():
            print(f"  {target:6.1f} m -> deeper than available GLORYS data")
        else:
            upper = depth[depth < target].max()
            lower = depth[depth > target].min()
            print(f"  {target:6.1f} m -> between {upper:.3f} m and {lower:.3f} m")

    print("\nVARIABLE ATTRIBUTES")
    for key, value in thetao.attrs.items():
        print(f"  {key:18s}: {value}")

    print("\nMISSING VALUE / ENCODING")
    for key in ["_FillValue", "missing_value", "dtype", "chunks", "compressor", "filters"]:
        print(f"  {key:18s}: {thetao.encoding.get(key)}")

    print("\nVALID DATA COVERAGE AT FIRST TIMESTEP")
    sample = thetao.isel(time=0)
    valid_count = sample.notnull().sum(dim=("latitude", "longitude")).values
    total_count = ds.sizes["latitude"] * ds.sizes["longitude"]
    valid_fraction = valid_count / total_count * 100

    for d, count, fraction in zip(depth, valid_count, valid_fraction):
        print(f"  {d:8.3f} m : {fraction:6.2f}% valid ({count:,}/{total_count:,})")

    print("\nDATASET SIZE")
    print(f"  thetao elements   : {thetao.size:,}")
    print(f"  nominal float size: {thetao.size * thetao.dtype.itemsize / 1024**3:.2f} GiB")

    ds.close()

In [ ]:
sst_regridded = xarray.open_zarr("../../data/processed/sst/sst_2015_2024_025deg.zarr")
glorys_regridded = xarray.open_zarr("../../data/processed/glorys/glorys_temperature_2015_2024_025deg.zarr")
sealevel_regridded = xarray.open_zarr("../../data/processed/sealevel/sealevel_2015_2024_025deg.zarr")
sss_regridded = xarray.open_zarr("../../data/processed/sss/sss_2015_2024_025deg.zarr")
wind_regridded = xarray.open_zarr("../../data/processed/wind/wind_2015_2024_025deg.zarr")

glorys_regridded.info()

In [2]:
surface_mask = xarray.open_zarr("../../data/processed/surface_mask_025deg.zarr")

In [13]:
surface_mask["surface_mask"].values

array([[False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]], shape=(101, 241))